# 論文を執筆する
論文の執筆を行うタスクです。<br>
論拠データや図表を使用して論文の執筆を行ってください。<br>
リサーチフローではLaTex（文章作成ツール）を使用して論文を執筆することが可能です。<br>
LaTexを使用する場合は後述の「リサーチフロー上で執筆する」の説明に従って論文の執筆、保存を行ってください。<br>
それ以外の方法で執筆する場合は論文の執筆後、「論文を執筆する」セルを実行してリサーチフロー上に直接アップロードしてください。<br>
<br>
論文を執筆する際にリサーチフローに保存した草稿や論拠データを閲覧したい場合は以下のセルを実行して下さい。

In [ ]:
# データを閲覧する
import os

from IPython.core.display import Javascript
from IPython.display import display
import panel as pn

from library.utils.access import open_data_folder
from library.utils.config import message as msg_config

button = open_data_folder(os.path.abspath('__file__'), 'argument_data',
                          button_name=msg_config.get('task', 'access_argument_data'))

pn.extension()
display(button)
display(Javascript('IPython.notebook.save_checkpoint();'))

## リサーチフロー上で執筆する
画像の説明に従って「論文を執筆する」セルから論文ファイルの作成、編集を行ってください。<br>
![ファイルを作成](./images/RF004010_create_paper.png)<br>
![ファイルを改名](./images/RF004010_rename_paper.png)<br>
<span style="color:red">※ 論文ファイルの編集後は必ず保存してからファイルを閉じてください。</span>

必要であれば論文に使用する図表を作成することができます。<br>
新しい図表を作成する場合は「図表を新たに作成する」ボタンを押下することで図表作成用のノートブックが作成されます。<br>
<br>
既に作成した図表の編集を行いたい場合は以下の手順に従ってください。<br>
①「2.図表を編集する」セルを実行して、図表と図表生成用のノートブックが保存されているフォルダを表示する。<br>
② 編集したい図表、またはノートブックを開いて編集する。<br>
<br>
また、リサーチフローを用いずに図表を作成した場合は以下の手順でアップロードを行ってください。<br>
①「論拠データを整理する」タスクに戻り「 2. データをアップロードする」セルを実行する。<br>
②アップロードしたいデータを選択してfigureディレクトリにアップロードする。<br>

### 1. 図表を新たに作成する

In [ ]:
# 図表を新たに作成する
import os
import re
import shutil

import panel as pn
from IPython.core.display import Javascript
from param.parameterized import Event

from library.task_director import TaskDirector
from library.utils.access import open_data_file
from library.utils.config import message as msg_config
from library.utils.setting import get_data_dir
from library.utils.widgets import Button, MessageBox


notebook_name = 'write_paper.ipynb'
template_notebook_path = '/home/jovyan/data_governance/base/task/writing/figure_template.ipynb'

class CreateFigureNoteBook(TaskDirector):
    """図表用ノートブックを作成するクラスです。

    Attributes:
        instance:
            working_path(str): 実行Notebookファイルパス
            output_message(MessageBox): メッセージ出力用のボックス
            form_section(pn.WidgetBox): ボタン等出力用のボックス
            text_input(pn.widgets.TextInput): ノートブック名を入力するためのフォーム
            generate_button_title(str): ノートブック作成用ボタンのタイトル文
            generte_button(Botton): ノートブック作成時に押下するボタン

    """
    def __init__(self, working_path: str):
        """CreateFigureNoteBookクラスのコンストラクタです。

        Args:
            working_path (str): 実行Notebookファイルパス

        """
        self.working_path = working_path
        super().__init__(self.working_path,  notebook_name)

        # メッセージ出力用のボックス
        self.output_message = MessageBox()
        self.output_message.width = 900
        # ボタン等表示用のボックス
        self.form_section = pn.WidgetBox()

    @TaskDirector.task_cell("3")
    def generate_name_form(self):
        """図表用ノートブックの名前を入力し、作成するためのフォームを作成するメソッドです。"""
        # 名前入力用フォームの設定
        self.text_input = pn.widgets.TextInput(name=msg_config.get(
            'create_draft', 'create_notebook'), width=300, height=50)
        # 作成時に押下するボタンの設定
        self.generate_button_title = msg_config.get(
            'create_draft', 'create_notebook')
        self.generate_button = Button(width=500)
        self.generate_button.set_looks_init(self.generate_button_title)
        self.generate_button.on_click(self.generate)
        self.generate_button.disabled = True

        def callback_input(event: Event):
            value = self.text_input.value_input
            if value is None or len(value) < 1:
                self.generate_button.disabled = True
                return
            self.generate_button.disabled = False

        self.text_input.param.watch(callback_input, 'value_input')

        # 表示する
        pn.extension()
        self.form_section.append(self.text_input)
        self.form_section.append(self.generate_button)
        self.form_section.append(self.output_message)
        display(self.form_section)
        self.generate_button.disabled = True
        display(Javascript('IPython.notebook.save_checkpoint();'))

    @TaskDirector.callback_form('図表用ノートブックを作成する')
    def generate(self, event):
        """ボタン押下時に図表用ノートブックを作成し、表示するためのボタンを出力するメソッドです。"""
        notebook_name = self.text_input.value_input

        # 値が入力されていない場合
        if not notebook_name.strip():
            self.output_message.update_warning(
                msg_config.get('create_draft', 'name_no_value'))
            return

        if re.search(r'[\\:\*\?"<>\|]', notebook_name):
            self.output_message.update_warning(
                msg_config.get('create_draft', 'pattern_warning'))
            return

        full_file_name = notebook_name + ".ipynb"
        dir_name = os.path.join(get_data_dir(self.working_path), 'figure')
        output_file_path = os.path.join(dir_name, full_file_name)

        # 入力した値のファイルが既に存在する場合
        if os.path.exists(os.path.join(dir_name, full_file_name)):
            self.output_message.update_warning(
                msg_config.get('create_draft', 'file_existed_warning'))
            return

        # テンプレートをコピーして新しいノートブックを作成
        shutil.copy(template_notebook_path, output_file_path)

        self.output_message.update_success(
            msg_config.get("create_draft", "success_create"))
        button = open_data_file(
            self.working_path, os.path.join('figure', full_file_name))

        # 表示を切り替え
        self.form_section.clear()
        self.form_section.append(self.output_message)
        self.form_section.append(button)
        display(Javascript('IPython.notebook.save_checkpoint();'))

CreateFigureNoteBook(working_path=os.path.abspath('__file__')).generate_name_form()

### 2. 図表を編集する

In [ ]:
# 図表フォルダを表示する
import os

from IPython.core.display import Javascript
from IPython.display import display
import panel as pn

from library.utils.access import open_data_folder


folder_name = 'figure'
button = open_data_folder(os.path.abspath('__file__'), folder_name)

pn.extension()
display(button)
display(Javascript('IPython.notebook.save_checkpoint();'))

### 3. 論文を執筆する

In [ ]:
# 論文フォルダを表示する
import os

from IPython.core.display import Javascript
from IPython.display import display
import panel as pn

from library.task_director import TaskDirector
from library.utils.access import open_data_folder


notebook_name = 'write_paper.ipynb'

def access_paper_folder(working_path: str):
    """論文フォルダを表示するメソッドです。"""

    task_director = TaskDirector(working_path, notebook_name)
    task_director.doing_task()

    folder_name = 'paper'
    button = open_data_folder(os.path.abspath('__file__'), folder_name)

    pn.extension()
    display(button)
    display(Javascript('IPython.notebook.save_checkpoint();'))
    task_director.done_task()

access_paper_folder(os.path.abspath('__file__'))


### 4. 論文を出力する

作成したtex形式の草稿をPDF形式の論文として出力する。<br>
次のセルを実行することで作成した草稿から論文を出力できます。<br>
指定する草稿ファイルは現在のサブフローの「paper」フォルダ内のデータを指定してください。<br>
出力時に同名のPDFファイルが存在していた場合は上書きします。<br>
論文出力時に現在のサブフローの「argument_data」、「figure」内のデータがそれぞれ論拠データ、図表として関連付けられます。<br>
整理が必要な場合は事前に行ってください。<br>
また、terminalで以下のコマンドを実行することでtex形式のファイルを同じフォルダ内にPDF形式で出力することができます。<br>
xelatex {対象のファイルまでのパス}<br>
<span style="color:red">※　次のセルを実行するとGakuNin RDMとの同期が行われます。</span><br>

In [ ]:
# texファイルをPDFとして出力する
import os
import subprocess
import traceback
from pathlib import Path

import httpx
import panel as pn
from IPython.core.display import Javascript
from IPython.display import display, clear_output
from requests.exceptions import RequestException

from library.task_director import TaskDirector
from library.utils.config import connect as con_config
from library.utils.config import message as msg_config
from library.utils.error import ( UnusableVault, ProjectNotExist,
                                    UnauthorizedError, RepoPermissionError)
from library.utils.input import get_grdm_connection_parameters
from library.utils.research_flow_provenance.prov import ProvenanceManager
from library.utils.setting.research_flow_status import get_subflow_type_and_id
from library.utils.setting import get_data_dir
from library.utils.storage_provider import grdm
from library.utils.widgets import Button, MessageBox

notebook_name = 'write_paper.ipynb'

class CompilePaper(TaskDirector):
    """親となる実験サブフローと実行中のサブフローの論拠データのフォルダを表示するクラスです。

    Attributes:
        instance:
            working_path(str): 実行Notebookファイルパス
            _msg_output(MessageBox): メッセージ出力用のボックス
            _form_section(pn.WidgetBox): ボタン等の出力を格納するためのボックス

    """
    def __init__(self, working_path: str):
        """Displyクラスのコンストラクタです。

        Args:
            working_path (str): 実行Notebookファイルパス

        """
        self.working_path = working_path
        super().__init__(self.working_path, notebook_name)

        self.grdm_url = con_config.get('GRDM', 'BASE_URL')
        self.grdm = grdm.Grdm()

        pn.extension()

        # 出力用フォームの設定
        self._form_section = pn.WidgetBox()
        # 実行結果出力用メッセージボックスの設定
        self._msg_output = MessageBox()
        self._msg_output.width = 900
        # コンパイル用ボタン
        self.compile_button = Button(width=500)

        # コピー時の警告用
        self.warning_area = pn.Row()
        self.warning_area.styles = {'background': '#ffe5e5'}

    def get_grdm_params(self) -> tuple[str, str]:
        """GRDMのトークンとプロジェクトIDを取得するメソッドです。

        Returns:
            str:GRDMのトークンの値を返す。
            str:プロジェクトIDの値を返す。
        """
        token = ""
        project_id = ""
        try:
            token, project_id = get_grdm_connection_parameters(self.grdm_url)
        except UnusableVault as e:
            message = msg_config.get('form', 'no_vault')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except RepoPermissionError:
            message = msg_config.get('form', 'insufficient_permission')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except ProjectNotExist as e:
            self._msg_output.update_error(str(e))
            self.log.error(traceback.format_exc())
        except RequestException as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
        return token, project_id

    async def sync_grdm(self):
        """GRDMにサブフローデータを同期する関数です。"""

        try:
            data_dir = get_data_dir(self.working_path)
            await self.grdm.sync(
                    token=self.token,
                    base_url=self.grdm_url,
                    project_id=self.project_id,
                    abs_source=data_dir,
                    abs_root=self._abs_root_path
                )
        except UnauthorizedError:
            message = msg_config.get('form', 'token_unauthorized')
            self._msg_output.update_warning(message)
            self.log.warning(f'{message}\n{traceback.format_exc()}')
            return
        except httpx.RequestError as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
            return
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
            return

    def init_form(self):
        """フォームの初期化を行うメソッドです。"""
        # テキスト入力欄を表示する場所
        text_inputs_column = pn.Column()
        # 入力フォームの作成
        data_dir = get_data_dir(self.working_path)
        paper_dir = os.path.join(data_dir, "paper")
        subflow_option = {}
        subflow_option[msg_config.get('write_paper', 'selector_default')] = "default"
        for file in Path(paper_dir).rglob("*.tex"):
            subflow_option[os.path.relpath(file, paper_dir)] = file
        self.tex_file_selector = pn.widgets.Select(
            name=msg_config.get('write_paper', 'selector_title'),
            options=subflow_option,
            value="default"
        )
        text_inputs_column.append(self.tex_file_selector)

        # コンパイル用ボタン
        self.compile_button.set_looks_init(msg_config.get('write_paper', 'compile_pdf'))
        self.compile_button.on_click(self._handle_click)
        self.compile_button.disabled = True

        def update_button_enabled(event):
            if event.obj.value == "default":
                self.compile_button.disabled = True
            else:
                self.compile_button.disabled = False

        # 入力欄に変化があったときに監視する
        self.tex_file_selector.param.watch(update_button_enabled, 'value')

        self._form_section.clear()
        self._form_section.append(text_inputs_column)
        self._form_section.append(self.compile_button)
        self._form_section.append(self._msg_output)

    @TaskDirector.task_cell("5")
    async def generate_form(self):
        """コンパイルするファイルの情報を入力するフォームを出力する関数です。"""

        self.doing_task()
        try:
            self.token, self.project_id = self.get_grdm_params()
            if not self._msg_output.has_message():
                # サブフローid
                _, self.subflow_id = get_subflow_type_and_id(self.working_path)
                # フォームの初期化
                self.init_form()
            else:
                self._form_section.append(self._msg_output)

        except Exception:
            self._form_section.clear()
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

        self.done_task()
        clear_output()
        display(self._form_section)
        self.compile_button.disabled = True
        display(Javascript('IPython.notebook.save_checkpoint();'))

    async def _handle_click(self, event):
        """非同期処理を実行するための仲介メソッドです。"""
        try:
            await self._run_compile(event)
        except Exception as e:
            self._form_section.clear()
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

    async def _run_compile(self, event):
        """出力ボタンが押された際の処理です。"""
        self.compile_button.disabled = True
        self.compile_button.set_looks_processing(msg_config.get('write_paper', 'doing_compile'))

        tex_file_path = self.tex_file_selector.value

        if not os.path.exists(tex_file_path):
            self._msg_output.update_error(f"草稿ファイル：{tex_file_path}が存在しません。")
            self.compile_button.set_looks_init(msg_config.get('write_paper', 'compile_pdf'))
            self.compile_button.disabled = False
            return

        # 出力先が既に存在する場合、警告を行う
        pdf_path = Path(tex_file_path).with_suffix(".pdf")
        self.existed_files = []
        if os.path.exists(pdf_path):
            existed_message = msg_config.get('write_paper', 'warning_samefile')
            self.existed_files.append(pdf_path)

            # 警告表示パネルの作成
            warning_md = pn.pane.Markdown(f"{existed_message}\n\n{self.existed_files}", width=500)
            continue_button = pn.widgets.Button(
                name=msg_config.get('write_paper', 'continue_compile'),
                button_type="warning", width=200
            )
            cancel_button = pn.widgets.Button(
                name=msg_config.get('write_paper', 'cancel'),
                button_type="warning", width=200
            )

            async def on_continue(event):
                """続行するボタン押下時の処理です。"""
                self.warning_area.clear()
                await self._handle_click2(tex_file_path)  # 本処理へ

            async def on_cancel(event):
                self.warning_area.clear()
                self.init_form()

            continue_button.on_click(on_continue)
            cancel_button.on_click(on_cancel)

            self.warning_area.clear()
            self.warning_area.append(pn.Column(warning_md, pn.Row(continue_button, cancel_button, align=('center'))))
            self._form_section.append(self.warning_area)

        else:
            await self._continue_compile(tex_file_path)

    async def _handle_click2(self, tex_file_path):
        """非同期処理を実行するための仲介メソッドです。"""
        try:
            await self._continue_compile(tex_file_path)
        except Exception as e:
            self._form_section.clear()
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

    @TaskDirector.callback_form("texファイルをPDFとして出力する")
    async def _continue_compile(self, tex_file_path):
        """コンパイルの本処理です。"""
        try:
            dir_name = os.path.join(get_data_dir(self.working_path), 'paper')
            # PDF化するコマンド
            result = subprocess.run(
                ["xelatex", f"-output-directory={dir_name}", tex_file_path],
                capture_output=True,
                text=True,
                check=True
            )

            #同期を走らせる
            await self.sync_grdm()

            def list_all_files_absolute(dir_name: str):
                """ファイルを全探索する関数です。"""
                dir_path = os.path.join(get_data_dir(self.working_path), dir_name)

                def check_ignore(filepath: Path):
                    for parent in filepath.parents:
                        if parent.name == ".ipynb_checkpoints":
                            return False
                    return True

                return [str(p.resolve()) for p in Path(dir_path).rglob('*')
                        if p.is_file() and check_ignore(p)]

            # 来歴処理
            prov_manager = ProvenanceManager(self.token, self.grdm_url, self.project_id)
            if self.existed_files:
                await prov_manager.handle("File Delete", self.existed_files)

            pdf_path = Path(tex_file_path).with_suffix(".pdf")
            tex_list = [tex_file_path]
            argument_list = list_all_files_absolute("argument_data")
            figure_list = list_all_files_absolute("figure")
            await prov_manager.handle("File Compile", pdf_path, tex_list, argument_list, figure_list)

            # 初期化
            self.compile_button.set_looks_init(msg_config.get('write_paper', 'compile_pdf'))
            self.compile_button.disabled = False

            self.tex_file_selector.value = "default"
            self.tex_file_selector.param.trigger('value')

            readme_message = msg_config.get('organize_argument_data', 'readme_link')
            self._msg_output.update_success(f"{msg_config.get('write_paper', 'done_compile')}\n{readme_message}")

        except subprocess.CalledProcessError as e:
            self._form_section.clear()
            message = f"出力に失敗しました: {e.stdout}"
            self._msg_output.update_error(message)
            self.log.error(message)

        except Exception as e:
            self._form_section.clear()
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

await CompilePaper(os.path.abspath('__file__')).generate_form()

### 5. 来歴情報を追加する
コードセルを用いらずにファイルのコピーやアップロード、コンパイル等の操作を行った場合、来歴情報が記録されていません。<br>
次のセルを実行して、来歴情報の追加を行ってください。<br>
<span style="color:red">※　次のセルを実行するとGakuNin RDMとの同期が行われます。</span><br>

In [ ]:
# 来歴情報を追加する
import os
import traceback

import httpx
import panel as pn
from IPython.core.display import Javascript
from IPython.display import display
from requests.exceptions import RequestException

from library.task_director import TaskDirector
from library.utils.access import list_files_recursively
from library.utils.config import connect as con_config
from library.utils.config import path_config, message as msg_config
from library.utils.error import (NotFoundSubflowDataError, UnusableVault, ProjectNotExist,
                                    UnauthorizedError, RepoPermissionError)
from library.utils.input import get_grdm_connection_parameters
from library.utils.research_flow_provenance.prov import ProvenanceManager
from library.utils.setting.research_flow_status import get_subflow_type_and_id, ResearchFlowStatusOperater
from library.utils.setting import get_data_dir
from library.utils.storage_provider import grdm
from library.utils.widgets import Button, MessageBox


notebook_name = 'write_paper.ipynb'

class AddProvenanceData(TaskDirector):
    """親となる実験サブフローと実行中のサブフローの論拠データのフォルダを表示するクラスです。

    Attributes:
        instance:
            working_path(str): 実行Notebookファイルパス
            self.grdm_url(str): GakuninRDMのベースURL
            self.grdm（Grdm）: Grdmクラスのインスタンス
            _msg_output(MessageBox): メッセージ出力用のボックス
            _form_section(pn.WidgetBox): ボタン等の出力を格納するためのボックス
            provenance_form(pn.Column): 来歴情報入力フォーム用のカラム
            warning_area(pn.Row): 警告アラート表示用のウィジェット
            reserch_flow_status_operater(ResearchFlowStatusOperater):　ResearchFlowStatusOperaterクラスのインスタンス
            token(str): GakuninRDMのトークン
            project_id(str): プロジェクトID
            prov_manager(ProvManager): 来歴情報操作クラスのインスタンス
            phase_number(int): 実行中のフェーズ番号
            subflow_data(dict): サブフローの情報
            subflow_type(str): サブフロータイプ
            subflow_id(str): サブフローID

    """
    def __init__(self, working_path: str):
        """Displyクラスのコンストラクタです。

        Args:
            working_path (str): 実行Notebookファイルパス

        """
        self.working_path = working_path
        super().__init__(self.working_path, notebook_name)

        self.grdm_url = con_config.get('GRDM', 'BASE_URL')
        self.grdm = grdm.Grdm()

        pn.extension()

        # 出力用フォームの設定
        self._form_section = pn.WidgetBox()
        # 実行結果出力用メッセージボックスの設定
        self._msg_output = MessageBox()
        self._msg_output.width = 900

        # 来歴情報入力フォーム用カラム
        self.provenance_form = pn.Column()

        self.warning_area = pn.Row()
        self.warning_area.styles = {'background': '#ffe5e5'}

        abs_root = path_config.get_abs_root_form_working_dg_file_path(self.working_path)
        research_flow_status_file_path = path_config.get_research_flow_status_file_path(abs_root)
        self.reserch_flow_status_operater = ResearchFlowStatusOperater(research_flow_status_file_path)

    def get_grdm_params(self) -> tuple[str, str]:
        """GRDMのトークンとプロジェクトIDを取得するメソッドです。

        Returns:
            str:GRDMのトークンの値を返す。
            str:プロジェクトIDの値を返す。
        """
        token = ""
        project_id = ""
        try:
            token, project_id = get_grdm_connection_parameters(self.grdm_url)
        except UnusableVault as e:
            message = msg_config.get('form', 'no_vault')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except RepoPermissionError:
            message = msg_config.get('form', 'insufficient_permission')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except ProjectNotExist as e:
            self._msg_output.update_error(str(e))
            self.log.error(traceback.format_exc())
        except RequestException as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
        return token, project_id

    async def sync_grdm(self):
        """GRDMにサブフローデータを同期する関数です。"""

        try:
            data_dir = get_data_dir(self.working_path)
            abs_path =  os.path.abspath(data_dir)
            await self.grdm.sync(
                    token=self.token,
                    base_url=self.grdm_url,
                    project_id=self.project_id,
                    abs_source=abs_path,
                    abs_root=self._abs_root_path
                )
        except UnauthorizedError:
            message = msg_config.get('form', 'token_unauthorized')
            self._msg_output.update_warning(message)
            self.log.warning(f'{message}\n{traceback.format_exc()}')
            return
        except httpx.RequestError as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
            return
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
            return

    @TaskDirector.task_cell("6")
    async def generate_activity_selector(self):
        """データフォルダを表示するためのボタンを生成するメソッドです。"""

        self.doing_task()
        self._form_section.append(self._msg_output)
        display(self._form_section)

        try:
            self.token, self.project_id = self.get_grdm_params()

            # サブフローの情報を取得
            self.subflow_type, self.subflow_id = get_subflow_type_and_id(self.working_path)
            phase_map = { "publication": 5, "review": 4, "writing": 3}
            self.phase_number = phase_map[self.subflow_type]
            self.subflow_data = {}
            self.get_parent_info(self.phase_number, self.subflow_id)

            # 表示する実験サブフローが存在する場合のみ実行する
            if not self.subflow_data:
                self._msg_output.update_warning(msg_config.get("organize_argument_data", "not_found_previous_subflow"))
                self._form_section.append(self._msg_output)
                return

            # 同じ論文執筆サブフローを親とする査読サブフローが存在する場合は表示する
            if self.subflow_type == "review":
                try:
                    self.get_children_info(self.subflow_data["writing"], self.subflow_id)
                except NotFoundSubflowDataError:
                    pass

            # 同期処理
            self._msg_output.update_info(msg_config.get('sync', 'doing'))
            await self.sync_grdm()
            self._msg_output.update_success(msg_config.get('sync', 'success'))

            data_dir = get_data_dir(self.working_path)
            abs_path =  os.path.abspath(data_dir)

            # 来歴情報処理用のインスタンスを作成
            self.prov_manager = ProvenanceManager(self.token, self.grdm_url, self.project_id)

            # ファイルが存在するかの検証を行う
            _, error_files = self.prov_manager.check_file_exist(abs_path)

            # 存在しないファイルがある場合はアラートを表示する。
            if error_files:
                alert_message = (
                    f"{msg_config.get('edit_provenance', 'describe_delete')}\n"
                    f"{msg_config.get('edit_provenance', 'describe_change')}\n"
                    f"{msg_config.get('edit_provenance', 'alert_message')}"
                )
                alert_message_md = pn.pane.Markdown(
                    alert_message.replace("\n", "  \n"),
                    styles={'font-weight': 'bold', 'color': '#b71c1c', 'margin-bottom': '10px'}
                )
                alerts = [self.create_file_alert(path, ids) for path, ids in error_files.items()]
                alerts_panel = pn.Column(*alerts, max_height=300, scroll=True)

                alert_card = pn.Card(
                    pn.Column(alert_message_md, alerts_panel),
                    title=msg_config.get('edit_provenance', 'title_Alert'),
                    styles={
                        "background-color": "#fdecea"  # 薄赤背景
                    },
                    margin=10,
                    width=900,
                    scroll=True
                )
                self._form_section.append(alert_card)

            self.provenance_form.clear()

            activity_option = ["", "コピー", "編集", "コンパイル", "エクスポート", "アップロード"]
            self.activity_selector = pn.widgets.Select(
                name=f"{msg_config.get('edit_provenance', 'title_activity_selector')}",
                options=activity_option,
                value=""
            )

            self.form_widgets = {}

            def create_form_for_activity(activity: str):
                """アクティビティごとのフォームを作成する関数です。

                Args:
                    activity (str): 選択されたアクティビティ

                """
                self.provenance_form.clear()
                self.form_widgets.clear()

                dst_file_list = list_files_recursively(abs_path)
                dst_options = {f"{msg_config.get('edit_provenance', 'select')}": "default"}
                dst_options.update({f: os.path.join(abs_path, f) for f in dst_file_list})
                dst_selector = pn.widgets.Select(name=f"{msg_config.get('edit_provenance', 'select_dst')}（{abs_path})", options=dst_options, value="default")
                self.form_widgets['dst_selector'] = dst_selector
                self.provenance_form.append(dst_selector)

                if activity == "コンパイル":
                    paper_options = {f"{msg_config.get('edit_provenance', 'select')}": "default"}
                    paper_options.update({f: os.path.join(abs_path, f) for f in dst_file_list})
                    paper_selector = pn.widgets.Select(name=f"{msg_config.get('edit_provenance', 'select_draft')}", options=paper_options, value="default")

                    multi_options = {f: os.path.join(abs_path, f) for f in dst_file_list}
                    argument_selector = pn.widgets.MultiSelect(name=f"{msg_config.get('edit_provenance', 'select_argument_data')}", options=multi_options)
                    figure_selector = pn.widgets.MultiSelect(name=f"{msg_config.get('edit_provenance', 'select_figure')}", options=multi_options)
                    self.form_widgets.update({
                        'paper_selector': paper_selector,
                        'argument_selector': argument_selector,
                        'figure_selector': figure_selector
                    })
                    self.provenance_form.extend([paper_selector, argument_selector, figure_selector])

                    submit_button = self._add_edit_button()
                    self.provenance_form.append(submit_button)

                elif activity == "エクスポート":
                    multi_options = {f: os.path.join(abs_path, f) for f in dst_file_list}
                    export_selector = pn.widgets.MultiSelect(name=f"{msg_config.get('edit_provenance', 'select_export')}", options=multi_options)
                    self.form_widgets['export_selector'] = export_selector
                    self.provenance_form.append(export_selector)
                    submit_button = self._add_edit_button()
                    self.provenance_form.append(submit_button)

                elif activity == "アップロード":
                    text_input = pn.widgets.TextAreaInput(name=f"{msg_config.get('upload_provenance', 'default_upload')}", placeholder= msg_config.get('upload_provenance', 'default_upload'))
                    self.form_widgets['text_input'] = text_input
                    self.provenance_form.append(text_input)

                    submit_button = self._add_edit_button()
                    self.provenance_form.append(submit_button)

                elif activity in ["コピー", "編集"]:
                    subflow_option = {}
                    subflow_option[f"{msg_config.get('edit_provenance', 'select')}"] = "default"
                    # 自身のサブフロー情報
                    subflow_name = self.reserch_flow_status_operater.get_flow_name(self.phase_number, self.subflow_id)
                    subflow_option[f"{msg_config.get('organize_argument_data', f'title_{self.subflow_type}')}{subflow_name}"] = self.subflow_id
                    # 親サブフローの情報
                    for subflow_type, parents_list in self.subflow_data.items():
                        partial = {
                            f"{msg_config.get('organize_argument_data', f'title_{subflow_type}')}{key}": val
                            for key, val in parents_list.items()
                        }
                        subflow_option.update(partial)

                    subflow_selector = pn.widgets.Select(name=f"{activity}元サブフロー選択", options=subflow_option, value="default")
                    self.form_widgets['subflow_selector'] = subflow_selector
                    src_form = pn.Column()
                    self.form_widgets['src_form'] = src_form

                    def on_subflow_change(event):
                        """サブフローが切り替えられた際の処理です。"""
                        src_form.clear()
                        if event.new == "default":
                            return
                        if event.new == self.subflow_id:
                            src_dir_path = abs_path
                            src_file_list = dst_file_list
                        else:
                            old_part= os.path.join(self.subflow_type, self.subflow_id)
                            for subflow_type, sub_dict in self.subflow_data.items():
                                if event.new in sub_dict.values():
                                    found_type = subflow_type
                            new_part = os.path.join(found_type, event.new)
                            new_working_path = self.working_path.replace(old_part, new_part)
                            src_dir_path =  os.path.abspath(get_data_dir(new_working_path))
                            src_file_list = list_files_recursively(src_dir_path)

                        src_options = {f"{msg_config.get('edit_provenance', 'select')}": "default"}
                        src_options.update({f: os.path.join(src_dir_path, f) for f in src_file_list})
                        src_selector = pn.widgets.Select(name=f"{activity}元のファイルを選択する（{src_dir_path})", options=src_options, value="default")
                        self.form_widgets['src_selector'] = src_selector
                        src_form.append(src_selector)
                        submit_button = self._add_edit_button()
                        src_form.append(submit_button)

                    subflow_selector.param.watch(on_subflow_change, "value")
                    self.provenance_form.extend([subflow_selector, src_form])

            def on_activity_change(event):
                """アクティビティが切替えられた際の処理です。"""
                self._msg_output.clear()
                if event.new == "":
                    self.provenance_form.clear()
                else:
                    create_form_for_activity(event.new)

            self.activity_selector.param.watch(on_activity_change, "value")

        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

        self.done_task()
        # 表示する
        self._form_section.append(pn.pane.Markdown(f"## {msg_config.get('edit_provenance', 'title_add')}"))
        self._form_section.append(self.activity_selector)
        self._form_section.append(self.provenance_form)
        self._form_section.remove(self._msg_output)
        self._form_section.append(self._msg_output)
        display(Javascript('IPython.notebook.save_checkpoint();'))

    def _add_edit_button(self):
        """共通の追加ボタン生成処理"""
        self.edit_button = Button(width=500)
        self.edit_button.set_looks_init(msg_config.get('edit_provenance', 'add_button'))
        self.edit_button.on_click(self._handle_click)
        return self.edit_button

    async def _handle_click(self, event):
        """非同期処理を実行するための仲介メソッドです。"""
        await self.process_activity_form(event)

    @TaskDirector.callback_form("来歴情報を追加する。")
    async def process_activity_form(self, event):
        """アクティビティごとに入力された値を処理する関数です。"""
        self.edit_button.disabled = True
        self._msg_output.clear()
        try:
            activity = self.activity_selector.value
            fw = self.form_widgets
            dst_file = fw.get('dst_selector').value if 'dst_selector' in fw else None
            if dst_file == "default":
                self._msg_output.update_error(msg_config.get('edit_provenance', 'dst_not_selected'))
                self.edit_button.disabled = False
                return

            if activity == "コンパイル":
                paper_selector = fw.get('paper_selector').value if 'paper_selector' in fw else None
                argument_values = fw.get('argument_selector').value if 'argument_selector' in fw else []
                figure_values = fw.get('figure_selector').value if 'figure_selector' in fw else []

                if paper_selector == "default":
                    self._msg_output.update_error(msg_config.get('edit_provenance', 'src_not_selected'))
                    self.edit_button.disabled = False
                    return
                else:
                    draft_list = [paper_selector]
                    self.edit_button.set_looks_processing(msg_config.get('edit_provenance', 'doing_record'))
                    await self.prov_manager.handle("File Compile", dst_file, draft_list, argument_values, figure_values)

            elif activity == "エクスポート":
                export_values = fw.get('export_selector').value if 'export_selector' in fw else []
                if not export_values:
                    self._msg_output.update_error(msg_config.get('edit_provenance', 'src_not_selected'))
                    self.edit_button.disabled = False
                    return
                else:
                    self.edit_button.set_looks_processing(msg_config.get('edit_provenance', 'doing_record'))
                    await self.prov_manager.handle("File Export", dst_file=dst_file, src_files=export_values)

            elif activity == "アップロード":
                text_value = self.form_widgets['text_input'].value_input if 'text_input' in self.form_widgets else ""
                if not text_value.strip():
                    self._msg_output.update_error(msg_config.get('edit_provenance', 'no_upload_info'))
                    self.edit_button.disabled = False
                    return
                else:
                    self.edit_button.set_looks_processing(msg_config.get('edit_provenance', 'doing_record'))
                    upload_provenance = {dst_file :text_value}
                    await self.prov_manager.handle("File Upload", upload_provenance)

            elif activity == "コピー":
                src_file = fw.get('src_selector').value if 'src_selector' in fw else None
                if src_file == "default":
                    self._msg_output.update_error(msg_config.get('edit_provenance', 'src_not_selected'))
                    self.edit_button.disabled = False
                    return
                else:
                    self.edit_button.set_looks_processing(msg_config.get('edit_provenance', 'doing_record'))
                    copy_provenance = {dst_file :src_file}
                    await self.prov_manager.handle("File Copy", copy_provenance)

            elif activity == "編集":
                src_file = fw.get('src_selector').value if 'src_selector' in fw else None
                if src_file == "default":
                    self._msg_output.update_error(msg_config.get('edit_provenance', 'src_not_selected'))
                    self.edit_button.disabled = False
                    return
                else:
                    self.edit_button.set_looks_processing(msg_config.get('edit_provenance', 'doing_record'))
                    modify_provenance = {dst_file :src_file}
                    await self.prov_manager.handle("File Modify", modify_provenance)

            self.edit_button.set_looks_init(msg_config.get('edit_provenance', 'add_button'))
            self.edit_button.disabled = False
            self.activity_selector.value = ""

            readme_message = msg_config.get('organize_argument_data', 'readme_link')
            self._msg_output.update_success(f"{msg_config.get('edit_provenance', 'done_record')}\n{readme_message}")

        except FileNotFoundError as e:
            message = f"{str(e)}\n{msg_config.get('sync', 'retry')}"
            self.log.error(f'{message}\n{traceback.format_exc()}')
            self._msg_output.update_error(message)

        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

    def create_file_alert(self, file_path: str, ids: list):
        """ファイルが存在しない場合のアラートを作成する関数です。

        Args:
            file_path (str): アラート対象のファイルパス
            ids (list): そのファイルに関する来歴情報

        Returns:
            pn.column: アラート表示用のカラム
        """
        alert = pn.pane.Markdown(
            f"""ファイル: `{file_path}`""",
            styles={
                "background-color": "#f8d7da",
                "padding": "10px",
                "border-radius": "5px",
                "border": "1px solid #f5c6cb"
            },
            margin=10,
            width=600
        )
        delete_button = pn.widgets.Button(name="削除", button_type="danger", width=100, align='center')
        change_button = pn.widgets.Button(name="パス変更", button_type="primary", width=100, align='center')

        # 新しいパス入力用ウィジェット
        new_path_input = pn.widgets.TextInput(name=msg_config.get('edit_provenance', 'new_path'), visible=False, width=500)
        confirm_change_button = pn.widgets.Button(name="変更する", button_type="success", visible=False, width=100, align=('start', 'end'))

        # エラーメッセージ用Markdown（最初は非表示）
        error_message = pn.pane.Markdown("", styles={"color": "red", "margin-left": "10px"})

        # ボタン押下時の処理
        async def on_delete(event):
            """非同期処理用の仲介メソッドです。"""
            await _handle_delete(event)

        @TaskDirector.callback_form("来歴情報を削除する。")
        async def _handle_delete(event):
            """削除ボタンが押下された際の関数です。"""
            delete_button.disabled =True
            change_button.disabled = True
            new_path_input.visible = False
            confirm_change_button.visible = False
            error_message.object = ""
            try:
                delete_file = [file_path]
                await self.prov_manager.handle("File Delete", delete_file)
            except Exception:
                message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
                self._msg_output.update_error(message)
                self.log.error(message)

            alert.object = f"{file_path} を削除しました。"
            delete_button.visible = False
            change_button.visible = False

        def on_change(event):
            """変更用フォームを表示する関数です。"""
            new_path_input.visible = True
            confirm_change_button.visible = True
            change_button.disabled = True  # 重複押し防止

        async def on_confirm_change(event):
            """パス変更ボタンが押下された際の仲介メソッドです。"""
            if new_path_input.value.strip() == "":
                error_message.object = msg_config.get('edit_provenance', 'new_path')
            else:
                delete_button.disabled =True
                confirm_change_button.disabled = True
                error_message.object = ""
                await _handle_change(event)

        @TaskDirector.callback_form("来歴情報のパスを変更する。")
        async def _handle_change(event):
            """パス変更ボタンが押下された際の処理です。"""
            new_path = new_path_input.value.strip()
            try:
                await self.prov_manager.handle("Provenance Edit", new_path, ids)

            except FileNotFoundError:
                message = f"入力されたパス：{new_path}がGakuninRDM上に存在しません。\n{msg_config.get('sync', 'retry')}"
                error_message.object = message
                delete_button.disabled =False
                confirm_change_button.disabled = False
                error_message.object = ""
                return

            except Exception:
                message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
                self._msg_output.update_error(message)
                self.log.error(message)
                return
            alert.object = f"{file_path} のパスを {new_path} に変更しました"
            # ボタンや入力欄を隠す
            new_path_input.visible = False
            confirm_change_button.visible = False
            delete_button.visible = False
            change_button.visible = False

        delete_button.on_click(on_delete)
        change_button.on_click(on_change)
        confirm_change_button.on_click(on_confirm_change)

        # アラートとボタンを横並びに表示
        return pn.Column(
        pn.Row(alert, delete_button, change_button),
        pn.Row(new_path_input, confirm_change_button),
        error_message
        )

    def get_parent_info(self, phase_seq_number :int, current_subflow: str):
        """親サブフローとなっている実験サブフローを取得するメソッドです。

        Args:
            phase_seq_number(str): 現在のフェーズ番号
            current_subflow (str): 現在実行中のサブフローのID

        """
        # 全ての親サブフローIDを取得
        parent_ids = self.reserch_flow_status_operater.get_parent_ids(phase_seq_number, current_subflow) # 3:論文執筆フェーズのシーケンス番号

         # 全ての親サブフロー名を取得
        phase_seq_number -= 1
        for parent_id in parent_ids:
            # フェーズ１（plan)以外を対象
            while phase_seq_number > 1:
                try:
                    parent_name = self.reserch_flow_status_operater.get_flow_name(phase_seq_number, parent_id)
                    if phase_seq_number == 2:
                        if "experiment" not in self.subflow_data:
                            self.subflow_data["experiment"] = {parent_name:parent_id}
                        else:
                            self.subflow_data["experiment"][parent_name] = parent_id
                        break

                    elif phase_seq_number == 3:
                        if "writing" not in self.subflow_data:
                            self.subflow_data["writing"] = {parent_name:parent_id}
                        else:
                            self.subflow_data["writing"][parent_name] = parent_id

                    elif phase_seq_number == 4:
                        if "review" not in self.subflow_data:
                            self.subflow_data["review"] = {parent_name:parent_id}
                        else:
                            self.subflow_data["review"][parent_name] = parent_id

                    # 全ての親サブフローを取得するように再帰的に呼び出す
                    self.get_parent_info(phase_seq_number, parent_id)
                    break
                # phase_seq_numberが一致しなかった場合は一つ前のフェーズで探索する
                except NotFoundSubflowDataError:
                    phase_seq_number -= 1

        return

    def get_children_info(self, parent_subflow: dict, subflow_id: str):
        """子サブフローとなっているサブフロー情報を取得するメソッドです。

        Args:
            parent_subflow (dict): 親サブフローの情報
            subflow_id (str): 実行サブフローのID

        Raises:
            NotFoundSubflowDataError: 子サブフローが見つからないかった場合のエラー

        """
        child_subflow = {}
        # 全ての親サブフローの子サブフロー情報を取得
        for parent_id in parent_subflow.values():
            try:
                child_subflow.update(self.reserch_flow_status_operater.get_children_id_and_name(4, parent_id))

            except NotFoundSubflowDataError:
                continue

        # 取得した子サブフローから実行中のサブフローを取り除く
        if len(child_subflow) > 1:
            working_subflow_name = self.reserch_flow_status_operater.get_flow_name(4, subflow_id)
            del child_subflow[working_subflow_name]
            if "review" not in self.subflow_data:
                self.subflow_data["review"] = child_subflow
            else:
                self.subflow_data["review"].update(child_subflow)
            return

        else:
            raise NotFoundSubflowDataError

await AddProvenanceData(os.path.abspath('__file__')).generate_activity_selector()


### 6. 来歴情報を削除する
「来歴情報を追加する」セルを用いて来歴情報を追加した場合や来歴情報登録後にファイル名やフォルダ構造を編集した場合に誤った来歴情報が登録されている可能性があります。<br>
次のセルを実行して、不要な来歴情報の削除を行ってください。<br>
<span style="color:red">※　次のセルを実行するとGakuNin RDMとの同期が行われます。</span><br>

In [ ]:
# 来歴情報を削除する
import os
import traceback

import httpx
import panel as pn
from IPython.core.display import Javascript
from IPython.display import display
from requests.exceptions import RequestException

from library.task_director import TaskDirector
from library.utils.access import create_single_file_selector
from library.utils.config import connect as con_config
from library.utils.config import message as msg_config
from library.utils.error import (UnusableVault, ProjectNotExist,
                                    UnauthorizedError, RepoPermissionError)
from library.utils.input import get_grdm_connection_parameters
from library.utils.research_flow_provenance.prov import ProvenanceManager
from library.utils.setting import get_data_dir
from library.utils.storage_provider import grdm
from library.utils.widgets import Button, MessageBox


notebook_name = 'write_paper.ipynb'

class DeleteProvenanceData(TaskDirector):
    """親となる実験サブフローと実行中のサブフローの論拠データのフォルダを表示するクラスです。

    Attributes:
        instance:
            working_path(str): 実行Notebookファイルパス
            self.grdm_url(str): GakuninRDMのベースURL
            self.grdm（Grdm）: Grdmクラスのインスタンス
            _msg_output(MessageBox): メッセージ出力用のボックス
            _form_section(pn.WidgetBox): ボタン等の出力を格納するためのボックス
            file_provenance(pn.Column): 来歴情報入力フォーム用のカラム
            warning_area(pn.Row): 警告アラート表示用のウィジェット
            token(str): GakuninRDMのトークン
            project_id(str): プロジェクトID
            prov_manager(ProvManager): 来歴情報操作クラスのインスタンス
            all_entity(dict): 全エンティティの情報

    """
    def __init__(self, working_path: str):
        """Displyクラスのコンストラクタです。

        Args:
            working_path (str): 実行Notebookファイルパス

        """
        self.working_path = working_path
        super().__init__(self.working_path, notebook_name)

        self.grdm_url = con_config.get('GRDM', 'BASE_URL')
        self.grdm = grdm.Grdm()

        pn.extension()

        # 出力用フォームの設定
        self._form_section = pn.WidgetBox()
        # 実行結果出力用メッセージボックスの設定
        self._msg_output = MessageBox()
        self._msg_output.width = 900
        # ファイルの来歴情報表示用カラム
        self.file_provenance = pn.Column()

        self.warning_area = pn.Row()
        self.warning_area.styles = {'background': '#ffe5e5'}

    def get_grdm_params(self) -> tuple[str, str]:
        """GRDMのトークンとプロジェクトIDを取得するメソッドです。

        Returns:
            str:GRDMのトークンの値を返す。
            str:プロジェクトIDの値を返す。
        """
        token = ""
        project_id = ""
        try:
            token, project_id = get_grdm_connection_parameters(self.grdm_url)
        except UnusableVault as e:
            message = msg_config.get('form', 'no_vault')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except RepoPermissionError:
            message = msg_config.get('form', 'insufficient_permission')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except ProjectNotExist as e:
            self._msg_output.update_error(str(e))
            self.log.error(traceback.format_exc())
        except RequestException as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
        return token, project_id

    async def sync_grdm(self):
        """GRDMにサブフローデータを同期する関数です。"""

        try:
            data_dir = get_data_dir(self.working_path)
            abs_path =  os.path.abspath(data_dir)
            await self.grdm.sync(
                    token=self.token,
                    base_url=self.grdm_url,
                    project_id=self.project_id,
                    abs_source=abs_path,
                    abs_root=self._abs_root_path
                )
        except UnauthorizedError:
            message = msg_config.get('form', 'token_unauthorized')
            self._msg_output.update_warning(message)
            self.log.warning(f'{message}\n{traceback.format_exc()}')
            return
        except httpx.RequestError as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
            return
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
            return

    @TaskDirector.task_cell("7")
    async def generate_file_selector(self):
        """データフォルダを表示するためのボタンを生成するメソッドです。"""

        self.doing_task()
        self._form_section.append(self._msg_output)
        display(self._form_section)
        #サブフローのデータをGRDMに同期する
        try:
            self.token, self.project_id = self.get_grdm_params()
            self._msg_output.update_info(msg_config.get('sync', 'doing'))
            await self.sync_grdm()
            self._msg_output.update_success(msg_config.get('sync', 'success'))

            data_dir = get_data_dir(self.working_path)
            abs_path =  os.path.abspath(data_dir)

            self.prov_manager = ProvenanceManager(self.token, self.grdm_url, self.project_id)

            # ファイルが存在するかの検証を行う
            self.all_entity, error_files = self.prov_manager.check_file_exist(abs_path)

            if error_files:
                alert_message = (
                    f"{msg_config.get('edit_provenance', 'describe_delete')}\n"
                    f"{msg_config.get('edit_provenance', 'describe_change')}\n"
                    f"{msg_config.get('edit_provenance', 'alert_message')}"
                )
                alert_message_md = pn.pane.Markdown(
                    alert_message.replace("\n", "  \n"),  # Markdownで改行するには行末に2スペース+改行
                    styles={'font-weight': 'bold', 'color': '#b71c1c', 'margin-bottom': '10px'}
                )
                alerts = [self.create_file_alert(path, ids) for path, ids in error_files.items()]
                alerts_panel = pn.Column(*alerts, max_height=300, scroll=True)

                alert_card = pn.Card(
                    pn.Column(alert_message_md, alerts_panel),
                    title=msg_config.get('edit_provenance', 'title_Alert'),
                    styles={
                        "background-color": "#fdecea"  # 薄赤背景
                    },
                    margin=10,
                    width=900,
                    scroll=True
                )
                self._form_section.append(alert_card)

            # 全ファイルを選択可能な状態で表示する。
            self.selector, self.relative_path, self.checkbox_dict = create_single_file_selector(self.working_path)

            self.edit_button = Button(width=500)
            self.edit_button.set_looks_init(msg_config.get('edit_provenance', 'delete_button'))
            self.edit_button.on_click(self. _handle_click)

        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

        self.done_task()
        # 表示する
        self._form_section.append(self.selector)
        self._form_section.append(self.edit_button)
        self._form_section.remove(self._msg_output)
        self._form_section.append(self._msg_output)
        display(Javascript('IPython.notebook.save_checkpoint();'))

    async def _handle_click(self, event):
        """非同期処理を実行するための仲介メソッドです。"""
        await self._edit(event)

    async def _edit(self, event):
        """来歴情報を削除するためのフォームを表示する関数です。"""
        try:
            self.file_provenance.clear()
            self.selected_file = None
            for checkbox in self.checkbox_dict.values():
                if checkbox.value:
                    self.selected_file = os.path.abspath(checkbox.file_path)
                    break
            if not self.selected_file:
                self._msg_output.update_error(msg_config.get('edit_provenance', 'not_selected'))
                self.edit_button = False
                return

            src_files = None
            if self.selected_file in self.all_entity:
                src_files = self.prov_manager.get_activity_info(self.all_entity[self.selected_file])

            if not src_files:
                self._form_section.remove(self._msg_output)
                self._msg_output.update_warning(msg_config.get('edit_provenance', 'no_provenance'))
                self._form_section.append(self._msg_output)
                return

            async def create_provenance_selector_panel(src_files: dict):
                """来歴情報用のウィジェットを生成する関数です。

                Args:
                    src_files (dict): 関連情報を表示するファイルの情報

                Returns:
                    pn.column: 来歴情報を表示するフォーム

                """
                cards = []

                def make_activity_card(activity_uri: str, data: dict):
                    """来歴情報を表示するカラムを生成する関数です。

                    Args:
                        activity_uri (str): アクティビティのURI
                        data (dict): 来歴情報

                    Returns:
                        pn.column: 来歴情報を表示するカラム

                    """
                    type_name = data.get("type", "不明な種別")
                    label_list = data.get("label", [])

                    osfstorage = "osfstorage"
                    base_path = os.environ['HOME']
                    file_path = []
                    for label in label_list:
                        base_trimmed = os.path.relpath(label, osfstorage)
                        file_path.append(os.path.join(base_path, base_trimmed))

                    label_md = "\n".join(f"- ファイル: {str(l)}" for l in file_path)
                    body = pn.pane.Markdown(label_md, width=760)

                    delete_button = pn.widgets.Button(
                        name=msg_config.get('edit_provenance', 'delete_provenance'),
                        button_type="danger",
                        width=200,
                    )

                    async def on_delete(event):
                        """削除ボタンが押下された際の処理です。"""
                        await self._handle_click2(activity_uri, file_path)

                    delete_button.on_click(on_delete)

                    container = pn.Column(
                        pn.pane.Markdown(f"### {type_name}", style={"font-weight": "bold"}),
                        body,
                        pn.Spacer(height=5),
                        delete_button,
                        width=800,
                        styles={"padding": "10px", "border": "1px solid lightgray", "border-radius": "5px"},
                        background="#ffffff"
                    )

                    return container

                for activity_uri, data in src_files.items():
                    card = make_activity_card(activity_uri, data)
                    cards.append(card)

                # 全体パネル
                panel = pn.Column(
                    pn.pane.Markdown(f"## {msg_config.get('edit_provenance', 'delete_selsector')}", styles={'font-weight': 'bold'}),
                    pn.pane.Markdown(f"{msg_config.get('edit_provenance', 'alert_message')}", styles={'color': 'red'}),
                    *cards,
                    pn.Spacer(height=10),
                    width=900,
                    background="#f9f9f9",
                    styles={'border': '1px solid lightgray', 'padding': '15px', 'border-radius': '8px'}
                )

                return panel

            panel = await create_provenance_selector_panel(src_files)

            edit_title = pn.pane.Markdown(f"""
                ## {msg_config.get('edit_provenance', 'title_delete')}
                ### ファイル：`{self.selected_file}`
                """)
            self.file_provenance.append(edit_title)
            self.file_provenance.append(panel)

            self._msg_output.clear()
            self._form_section.append(self.file_provenance)
            self._form_section.remove(self._msg_output)
            self._form_section.append(self._msg_output)

        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
            return

    async def _handle_click2(self, activity_uri: str, src_path: list):
        """非同期処理を実行するための仲介メソッドです。

        Args:
            activity_uri (str): 削除するアクティビティのURI
            src_path (str): 関連情報を削除するファイルのパス

        """
        await self._delete(activity_uri, src_path)

    @TaskDirector.callback_form("来歴情報を削除する。")
    async def _delete(self, activity_uri: str, src_label: list):
        """来歴情報を削除する用の関数です。

        Args:
            activity_uri (str): 削除するアクティビティのURI
            src_label (str): 関連情報を削除するファイルのパス

        """
        try:
            self.file_provenance.clear()
            self._msg_output.update_info(msg_config.get('edit_provenance', 'doing_delete'))
            update_files = src_label
            update_files.append(self.selected_file)
            await self.prov_manager.handle("Delete Activity",activity_uri, update_files)

            readme_message = msg_config.get('organize_argument_data', 'readme_link')
            self._msg_output.update_success(f"{msg_config.get('edit_provenance', 'done_delete')}\n{readme_message}")
            self._form_section.append(self._msg_output)

        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
            return

    def create_file_alert(self, file_path: str, ids: list):
        """ファイルが存在しない場合のアラートを作成する関数です。

        Args:
            file_path (str): アラート対象のファイルパス
            ids (list): そのファイルに関する来歴情報

        Returns:
            pn.column: アラート表示用のカラム
        """
        alert = pn.pane.Markdown(
            f"""ファイル: `{file_path}`""",
            styles={
                "background-color": "#f8d7da",
                "padding": "10px",
                "border-radius": "5px",
                "border": "1px solid #f5c6cb"
            },
            margin=10,
            width=600
        )
        delete_button = pn.widgets.Button(name="削除", button_type="danger", width=100, align='center')
        change_button = pn.widgets.Button(name="パス変更", button_type="primary", width=100, align='center')

        # 新しいパス入力用ウィジェット
        new_path_input = pn.widgets.TextInput(name=msg_config.get('edit_provenance', 'new_path'), visible=False, width=500)
        confirm_change_button = pn.widgets.Button(name="変更する", button_type="success", visible=False, width=100, align=('start', 'end'))

        # エラーメッセージ用Markdown（最初は非表示）
        error_message = pn.pane.Markdown("", styles={"color": "red", "margin-left": "10px"})

        # ボタン押下時の処理
        async def on_delete(event):
            """非同期処理用の仲介メソッドです。"""
            await _handle_delete(event)

        @TaskDirector.callback_form("来歴情報を削除する。")
        async def _handle_delete(event):
            """削除ボタンが押下された際の関数です。"""
            delete_button.disabled =True
            change_button.disabled = True
            new_path_input.visible = False
            confirm_change_button.visible = False
            error_message.object = ""
            try:
                delete_file = [file_path]
                await self.prov_manager.handle("File Delete", delete_file)
            except Exception:
                message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
                self._msg_output.update_error(message)
                self.log.error(message)

            alert.object = f"{file_path} を削除しました。"
            delete_button.visible = False
            change_button.visible = False

        def on_change(event):
            """変更用フォームを表示する関数です。"""
            new_path_input.visible = True
            confirm_change_button.visible = True
            change_button.disabled = True  # 重複押し防止

        async def on_confirm_change(event):
            """パス変更ボタンが押下された際の仲介メソッドです。"""
            if new_path_input.value.strip() == "":
                error_message.object = msg_config.get('edit_provenance', 'new_path')
            else:
                delete_button.disabled =True
                confirm_change_button.disabled = True
                error_message.object = ""
                await _handle_change(event)

        @TaskDirector.callback_form("来歴情報のパスを変更する。")
        async def _handle_change(event):
            """パス変更ボタンが押下された際の処理です。"""
            new_path = new_path_input.value.strip()
            try:
                await self.prov_manager.handle("Provenance Edit", new_path, ids)

            except FileNotFoundError:
                message = f"入力されたパス：{new_path}がGakuninRDM上に存在しません。\n{msg_config.get('sync', 'retry')}"
                error_message.object = message
                delete_button.disabled =False
                confirm_change_button.disabled = False
                error_message.object = ""
                return

            except Exception:
                message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
                self._msg_output.update_error(message)
                self.log.error(message)
                return
            alert.object = f"{file_path} のパスを {new_path} に変更しました"
            # ボタンや入力欄を隠す
            new_path_input.visible = False
            confirm_change_button.visible = False
            delete_button.visible = False
            change_button.visible = False

        delete_button.on_click(on_delete)
        change_button.on_click(on_change)
        confirm_change_button.on_click(on_confirm_change)

        # アラートとボタンを横並びに表示
        return pn.Column(
        pn.Row(alert, delete_button, change_button),
        pn.Row(new_path_input, confirm_change_button),
        error_message
        )

await DeleteProvenanceData(os.path.abspath('__file__')).generate_file_selector()


## GakuNin RDMに保存する

In [ ]:
# GakuNin RDMに保存する
import os

import panel as pn
from IPython.core.display import Javascript
from IPython.display import display

from library.task_director import TaskDirector
from library.utils.research_flow_provenance import ProvenanceEditor
from library.utils.setting import get_data_dir

script_file_name = 'write_paper'
notebook_name = script_file_name+'.ipynb'


class DataSaver(TaskDirector):
    """GRDMに保存するクラスです。

    Attributes:
        instance:
            _abs_root_path (str): 絶対パス
            save_form_box(pn.WidgetBox):フォームを格納する。
            save_msg_output(Message):ユーザーに提示するメッセージを格納する。
    """
    def __init__(self, working_path: str) -> None:
        """DataSaver コンストラクタメソッドです。

        Args:
            working_path (str): 実行Notebookファイルパス

        """
        self.working_path = working_path
        super().__init__(self.working_path, notebook_name)

    @TaskDirector.task_cell("6")
    def generate_form_section(self):
        """取得したデータを表示するメソッドです。"""
        # タスク開始によるサブフローステータス管理JSONの更新

        # フォーム定義
        data_dir = get_data_dir(self.working_path)
        source = [os.path.join(data_dir, 'figure'), os.path.join(data_dir, 'paper')]
        sync_files = ProvenanceEditor().get_file_list()
        sync_files.append(os.path.join(data_dir, 'README.md'))
        for file in sync_files:
            if os.path.exists(file):
                source.append(file)
        self.define_save_form(source)
        # フォーム表示
        pn.extension()
        form_section = pn.WidgetBox()
        form_section.append(self.save_form_box)
        form_section.append(self.save_msg_output)
        display(form_section)
        display(Javascript('IPython.notebook.save_checkpoint();'))

DataSaver(working_path=os.path.abspath('__file__')).generate_form_section()

## サブフローメニューを表示する

In [ ]:
# サブフローメニューを表示する
import os

from library.task_director import TaskDirector

script_file_name = "write_paper"
notebook_name = script_file_name+'.ipynb'
task_director = TaskDirector(os.path.abspath('__file__'), notebook_name)
task_director.return_subflow_menu()